
读取 `data/` 下的原始文件，查看文件列表、样例行、列类型与缺失值统计。


In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from config import CSV_CHUNKSIZE, DATA_ROOT, RAW_OHLC_CSV
from data.prepare import OHLC_DTYPES

RAW_CSV = RAW_OHLC_CSV
CHUNKSIZE = CSV_CHUNKSIZE

In [2]:
raw_files = sorted(p for p in (DATA_ROOT / "raw").rglob("*") if p.is_file())
processed_files = sorted(p for p in (DATA_ROOT / "processed").rglob("*") if p.is_file())

print(f"data root: {DATA_ROOT}\n")
print("raw files:")
for p in raw_files:
    size_mb = p.stat().st_size / 1024**2
    print(f"  {p.relative_to(DATA_ROOT)}  ({size_mb:,.1f} MB)")

print("\nprocessed files:")
if processed_files:
    for p in processed_files:
        size_mb = p.stat().st_size / 1024**2
        print(f"  {p.relative_to(DATA_ROOT)}  ({size_mb:,.1f} MB)")
else:
    print("  (empty)")

data root: /home/zkb/rpt-re/data

raw files:
  raw/OHLC_92_24.csv  (6,567.0 MB)

processed files:
  (empty)


In [3]:
head = pd.read_csv(RAW_CSV, nrows=5, dtype=OHLC_DTYPES)
head

,PERMNO,HdrCUSIP,Ticker,PERMCO,DlyCalDt,DlyCap,DlyRet,DlyRetx,DlyVol,DlyClose,DlyLow,DlyHigh,DlyOpen
0,10001,36720410,GFGC,7953,1992-01-02,15587.50,0.000000,0.000000,100.0,14.500,14.5,14.500,NaN
1,10001,36720410,GFGC,7953,1992-01-03,15587.50,0.000000,0.000000,498.0,14.500,14.5,14.500,NaN
2,10001,36720410,GFGC,7953,1992-01-06,15587.50,0.000000,0.000000,100.0,14.500,14.5,14.500,NaN
3,10001,36720410,GFGC,7953,1992-01-07,15587.50,0.000000,0.000000,417.0,14.500,14.5,15.250,NaN
4,10001,36720410,GFGC,7953,1992-01-08,16259.38,0.043103,0.043103,500.0,15.125,14.5,15.125,NaN


CRSP 日频数据：`PERMNO` 是 CRSP 给每只股票分配的永久编号，`HdrCUSIP` 是对应的 CUSIP 证券代码，`Ticker` 是股票代码（这里是 GFGC），`PERMCO` 是公司层面的永久编号；`DlyCalDt` 是交易日，`DlyCap` 是当日市值，`DlyRet` 是含分红再投资的日收益率，`DlyRetx` 是不含分红的日收益率，`DlyVol` 是成交量，`DlyClose`、`DlyLow`、`DlyHigh`、`DlyOpen` 分别是收盘价、最低价、最高价和开盘价。

In [4]:
sample = pd.read_csv(RAW_CSV, nrows=10_000, dtype=OHLC_DTYPES)

print(f"columns ({len(sample.columns)}): {list(sample.columns)}\n")
print(sample.dtypes)

print("\n--- numeric describe (sample) ---")
display(sample.describe())

print("\n--- date range (sample) ---")
print(f"min: {sample['DlyCalDt'].min()}")
print(f"max: {sample['DlyCalDt'].max()}")

print("\n--- cardinality (sample) ---")
print(f"unique PERMNO: {sample['PERMNO'].nunique():,}")
print(f"unique Ticker: {sample['Ticker'].nunique():,}")

columns (13): ['PERMNO', 'HdrCUSIP', 'Ticker', 'PERMCO', 'DlyCalDt', 'DlyCap', 'DlyRet', 'DlyRetx', 'DlyVol', 'DlyClose', 'DlyLow', 'DlyHigh', 'DlyOpen']

PERMNO        int64
HdrCUSIP     string
Ticker       string
PERMCO        int64
DlyCalDt     string
DlyCap      float64
DlyRet      float64
DlyRetx     float64
DlyVol      float64
DlyClose    float64
DlyLow      float64
DlyHigh     float64
DlyOpen     float64
dtype: object

--- numeric describe (sample) ---


,PERMNO,PERMCO,DlyCap,DlyRet,DlyRetx,DlyVol,DlyClose,DlyLow,DlyHigh,DlyOpen
count,10000.000000,10000.000000,9999.000000,10000.000000,10000.000000,9.999000e+03,7989.000000,7989.000000,7989.000000,7898.000000
mean,10001.355200,7953.355200,60678.832784,0.000798,0.000645,9.292006e+03,11.235649,11.094274,11.366250,11.216042
std,0.478598,0.478598,50681.470745,0.027874,0.027883,3.124403e+04,3.652046,3.608254,3.689201,3.662207
min,10001.000000,7953.000000,7050.000000,-0.174150,-0.174150,0.000000e+00,5.100000,4.740000,5.500000,5.120000
25%,10001.000000,7953.000000,20510.250000,-0.009704,-0.009791,2.000000e+02,8.750000,8.650000,8.850000,8.750000
50%,10001.000000,7953.000000,39361.880000,0.000000,0.000000,2.000000e+03,10.260000,10.140000,10.380000,10.240000
75%,10002.000000,7954.000000,91323.265000,0.010897,0.010627,8.400000e+03,12.700000,12.625000,12.845000,12.700000
max,10002.000000,7954.000000,271226.810000,0.647135,0.647135,1.495729e+06,31.000000,31.000000,31.000000,31.000000



--- date range (sample) ---
min: 1992-01-02
max: 2017-08-04

--- cardinality (sample) ---
unique PERMNO: 2
unique Ticker: 6


In [5]:
row_count = sum(
    len(chunk)
    for chunk in pd.read_csv(
        RAW_CSV,
        chunksize=CHUNKSIZE,
        usecols=["PERMNO"],
        dtype={"PERMNO": "int64"},
    )
)
print(f"total rows: {row_count:,}")

total rows: 64,195,103


In [6]:
total_rows = 0
null_counts = None
for chunk in pd.read_csv(RAW_CSV, chunksize=CHUNKSIZE, dtype=OHLC_DTYPES):
    total_rows += len(chunk)
    chunk_nulls = chunk.isna().sum()
    null_counts = chunk_nulls if null_counts is None else null_counts + chunk_nulls

missing_stats = pd.DataFrame(
    {
        "null_count": null_counts,
        "null_pct": null_counts / total_rows * 100,
    }
)
missing_stats.sort_values("null_count", ascending=False)

,null_count,null_pct
DlyOpen,4646238,7.237683
DlyHigh,4010680,6.247642
DlyClose,4010678,6.247639
DlyLow,4010678,6.247639
DlyCap,883869,1.376848
DlyRetx,883407,1.376128
DlyRet,883407,1.376128
DlyVol,879642,1.370263
Ticker,849622,1.323500
DlyCalDt,0,0.000000
